In [1]:
import pandas as pd

df = pd.read_csv("../outputs/cleaned_transactions.csv")
df.head()

,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,...,errors,error_flag,refund_flag,year,month,month_name,quarter,weekday,hour,weekend_flag
0,11978328,2012-11-23 16:03:00,619,3348,263.43,Swipe Transaction,54850,Austin,MN,55912.0,...,NaN,0,0,2012,11,November,4,Friday,16,0
1,11363233,2012-07-07 13:57:00,456,4576,38.26,Swipe Transaction,68135,Cape Coral,FL,33909.0,...,NaN,0,0,2012,7,July,3,Saturday,13,1
2,8117710,2010-06-10 16:59:00,209,4676,52.57,Swipe Transaction,81833,El Paso,TX,79928.0,...,NaN,0,0,2010,6,June,2,Thursday,16,0
3,12606562,2013-04-13 12:08:00,1605,1133,40.00,Swipe Transaction,27092,Amelia,OH,45102.0,...,NaN,0,0,2013,4,April,2,Saturday,12,1
4,12628171,2013-04-18 10:00:00,144,5247,4.58,Swipe Transaction,44578,Arkadelphia,AR,71923.0,...,NaN,0,0,2013,4,April,2,Thursday,10,0


In [2]:
print(df.columns.tolist())

['id', 'date', 'client_id', 'card_id', 'amount', 'use_chip', 'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors', 'error_flag', 'refund_flag', 'year', 'month', 'month_name', 'quarter', 'weekday', 'hour', 'weekend_flag']


In [3]:
df['error_flag'].value_counts()

error_flag
0    49219
1      781
Name: count, dtype: int64

In [4]:
df['error_flag'].value_counts(normalize=True) * 100

error_flag
0    98.438
1     1.562
Name: proportion, dtype: float64

In [5]:
df.isnull().sum().sort_values(ascending=False)

errors            49219
zip                6148
merchant_state     5802
date                  0
id                    0
amount                0
card_id               0
client_id             0
use_chip              0
merchant_city         0
merchant_id           0
mcc                   0
error_flag            0
refund_flag           0
year                  0
month                 0
month_name            0
quarter               0
weekday               0
hour                  0
weekend_flag          0
dtype: int64

In [6]:
for col in ['use_chip','weekday','quarter']:
    print(f"\n{col}")
    print(df[col].value_counts())


use_chip
use_chip
Swipe Transaction     26224
Chip Transaction      18000
Online Transaction     5776
Name: count, dtype: int64

weekday
weekday
Tuesday      7249
Thursday     7229
Saturday     7213
Sunday       7173
Monday       7080
Wednesday    7031
Friday       7025
Name: count, dtype: int64

quarter
quarter
3    12849
2    12671
1    12595
4    11885
Name: count, dtype: int64


In [7]:
features = [
    'amount',
    'hour',
    'month',
    'quarter',
    'weekend_flag',
    'use_chip',
    'weekday',
    'mcc',
    'merchant_state'
]

ml_df = df[features + ['error_flag']].copy()

print(ml_df.shape)
print("\nMissing Values:\n")
print(ml_df.isnull().sum())

(50000, 10)

Missing Values:

amount               0
hour                 0
month                0
quarter              0
weekend_flag         0
use_chip             0
weekday              0
mcc                  0
merchant_state    5802
error_flag           0
dtype: int64


In [12]:
!pip install scikit-learn

  Using cached scipy-1.17.1-cp314-cp314-win_amd64.whl.metadata (60 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.3 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.3 MB 758.3 kB/s eta 0:00:11
   -- ------------------------------------- 0.5/8.3 MB 758.3 kB/s eta 0:00:11
   -- ------------------------------------- 0.5/8.3 MB 758.3 kB/s eta 0:00:11
   --- ------------------------------------ 0.8/8.3 MB 514.5 kB/s eta 0:00:15
   --- ------------------------------------ 0.8/8.3 MB 514.5 kB/s eta 0:00:15
   ----- ---------------------------------- 1.0/8.3 MB 582.7 kB/s eta 0:00:13
   ------ --------------------------------- 1.3/8.3 M


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
from sklearn.model_selection import train_test_split

ml_df = ml_df.copy()

# Replace missing states
ml_df['merchant_state'] = ml_df['merchant_state'].fillna('Unknown')

X = ml_df.drop('error_flag', axis=1)
y = ml_df['error_flag']

print(X.shape)
print(y.shape)

(50000, 9)
(50000,)


In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

numeric_features = [
    'amount',
    'hour',
    'month',
    'quarter',
    'weekend_flag'
]

categorical_features = [
    'use_chip',
    'weekday',
    'mcc',
    'merchant_state'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42
    ))
])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(40000, 9)
(10000, 9)


In [15]:
model.fit(X_train, y_train)

print("Training Complete")

Training Complete


In [16]:
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.59      0.74      9844
           1       0.02      0.56      0.04       156

    accuracy                           0.59     10000
   macro avg       0.50      0.57      0.39     10000
weighted avg       0.97      0.59      0.73     10000


Confusion Matrix:
[[5776 4068]
 [  69   87]]


In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

rf_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

print("Random Forest Trained")

Random Forest Trained


In [18]:
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

rf_pred = rf_model.predict(X_test)

print(classification_report(y_test, rf_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_pred))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99      9844
           1       0.00      0.00      0.00       156

    accuracy                           0.98     10000
   macro avg       0.49      0.50      0.50     10000
weighted avg       0.97      0.98      0.98     10000


Confusion Matrix:
[[9827   17]
 [ 156    0]]


In [19]:
risk_scores = model.predict_proba(X)[:, 1]

ml_df['risk_score'] = risk_scores

In [20]:
ml_df['risk_category'] = pd.cut(
    ml_df['risk_score'],
    bins=[0, 0.3, 0.7, 1],
    labels=['Low Risk', 'Medium Risk', 'High Risk']
)

In [21]:
ml_df['risk_category'].value_counts()

risk_category
Medium Risk    37564
Low Risk        9653
High Risk       2783
Name: count, dtype: int64

In [22]:
prediction_df = df.copy()

prediction_df['risk_score'] = model.predict_proba(X)[:, 1]

prediction_df['risk_category'] = pd.cut(
    prediction_df['risk_score'],
    bins=[0, 0.3, 0.7, 1],
    labels=['Low Risk', 'Medium Risk', 'High Risk']
)

prediction_df.to_csv(
    'transaction_risk_predictions.csv',
    index=False
)

print("File Saved Successfully")

File Saved Successfully
